# 09 — Sales vs Profit by State
Top 20 states by revenue, with profit bars side-by-side for quick comparison.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Sales vs Profit — Top 20 States ──────────────────────────────────────────
states = fetch_df(cur, """
    SELECT l.state,
           ROUND(SUM(oi.sales),  2) AS sales,
           ROUND(SUM(oi.profit), 2) AS profit
    FROM orders o
    JOIN order_items oi ON o.order_id    = oi.order_id
    JOIN locations   l  ON o.postal_code = l.postal_code
    GROUP BY l.state
    ORDER BY sales DESC
    LIMIT 20
""")

y     = range(len(states))
bar_w = 0.38

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh([i + bar_w/2 for i in y], states["sales"][::-1],
        bar_w, color=PALETTE[0], label="Sales",  edgecolor="white")
ax.barh([i - bar_w/2 for i in y], states["profit"][::-1],
        bar_w, color=PALETTE[1], label="Profit", edgecolor="white")

ax.set_yticks(list(y))
ax.set_yticklabels(states["state"][::-1], fontsize=9)

for i, (sal, prof) in enumerate(zip(states["sales"][::-1],
                                     states["profit"][::-1])):
    ax.text(sal + 200, i + bar_w/2, fmt_usd(sal),
            va="center", fontsize=7.5)
    ax.text(prof + 200 if prof >= 0 else prof - 200,
            i - bar_w/2, fmt_usd(prof),
            va="center", fontsize=7.5,
            ha="left" if prof >= 0 else "right")

ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.legend(frameon=False)
ax.set_xlabel("USD")
ax.set_title("Sales vs Profit for Top 20 States")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
